# Reply Mirror — Fraud Detection Multi-Agent System
### LangChain `create_agent` pattern + OpenRouter + Langfuse

## Cell 1 — Install
Pin `langchain==1.0.0` which is the version that ships `create_agent` in `langchain.agents`.

In [3]:
!pip install -q "langchain==1.0.0" langchain-core langchain-openai langchain-community langfuse ulid-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.4/476.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.

## Cell 2 — Secrets

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["OPENROUTER_API_KEY"]  = s.get_secret("OPENROUTER_API_KEY")
    os.environ["LANGFUSE_PUBLIC_KEY"] = s.get_secret("LANGFUSE_PUBLIC_KEY")
    os.environ["LANGFUSE_SECRET_KEY"] = s.get_secret("LANGFUSE_SECRET_KEY")
    os.environ["LANGFUSE_HOST"]       = s.get_secret("LANGFUSE_HOST")
    os.environ["TEAM_NAME"]           = s.get_secret("TEAM_NAME")
    print("Secrets loaded from Kaggle.")
except Exception as e:
    print(f"Using defaults ({e})")


Secrets loaded from Kaggle.


In [ ]:
print("HOST  :", os.environ.get("LANGFUSE_HOST", "NOT SET"))
print("PUBLIC:", os.environ.get("LANGFUSE_PUBLIC_KEY", "NOT SET")[:20] + "...")
print("SECRET:", os.environ.get("LANGFUSE_SECRET_KEY", "NOT SET")[:20] + "...")
print("TEAM  :", os.environ.get("TEAM_NAME", "NOT SET"))
print("OPENR :", os.environ.get("OPENROUTER_API_KEY", "NOT SET")[:20] + "...")

## Cell 3 — Imports + Model + Langfuse
Exact tutorial import pattern:
```python
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
```

In [8]:
import os, json, re, pandas as pd, ulid
from datetime import timedelta
from pathlib import Path

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# ── New langfuse import — compatible with langchain==1.0.0 ───────────────────
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler   # ← correct for new langfuse

import langchain
print("LangChain version:", langchain.__version__)

# ── Shared model ──────────────────────────────────────────────────────────────
model = ChatOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    model="openai/gpt-4o-mini",
    temperature=0.2,
    max_tokens=800,
)

# ── Langfuse client ───────────────────────────────────────────────────────────
langfuse_client = Langfuse(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    host=os.environ["LANGFUSE_HOST"],
)

def generate_session_id():
    team = os.environ.get("TEAM_NAME", "team").replace(" ", "-")
    return f"{team}-{ulid.new().str}"

# def get_callback_handler():
#     return CallbackHandler(
#         public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
#         secret_key=os.environ["LANGFUSE_SECRET_KEY"],
#         host=os.environ["LANGFUSE_HOST"],
#     )

def get_callback_handler():
    return CallbackHandler()

print("Model and Langfuse ready.")

LangChain version: 1.0.0
Model and Langfuse ready.


## Cell 4 — Data Loader

In [9]:
def load_dataset(dataset_path: str) -> dict:
    """
    Loads all files from a dataset folder.
    Returns dict: transactions, users, sms, mails, locations, audio_files, level
    """
    base = Path(dataset_path)
    subs = [f for f in base.iterdir() if f.is_dir() and not f.name.startswith("__")]
    if subs:
        base = subs[0]

    data = {}

    tx = base / "transactions.csv"
    if tx.exists():
        df = pd.read_csv(tx)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        data["transactions"] = df
    else:
        data["transactions"] = pd.DataFrame()

    for key in ["users", "sms", "mails", "locations"]:
        p = base / f"{key}.json"
        data[key] = json.load(open(p, encoding="utf-8")) if p.exists() else []

    audio_dir = base / "audio"
    data["audio_files"] = sorted(audio_dir.glob("*.mp3")) if audio_dir.exists() else []

    n = len(data["users"])
    data["level"] = (
        "deus_ex"         if data["audio_files"] else
        "truman_show"     if n <= 3             else
        "brave_new_world" if n <= 7             else
        "deus_ex"
    )

    print(f"Level        : {data['level']}")
    print(f"Transactions : {len(data['transactions'])}")
    print(f"Users        : {n}")
    print(f"Audio files  : {len(data['audio_files'])}")
    return data

## Cell 5 — Load Dataset
> Update `DATASET_PATH` to your Kaggle input folder.

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/katayounkobraei/brave-new-world/Brave New World"

DATA = load_dataset(DATASET_PATH)

Level        : brave_new_world
Transactions : 524
Users        : 7
Audio files  : 0


## Cell 6 — Specialist Agents
Each specialist uses `create_agent(model=model, system_prompt="...")` — no tools.
Short prompts to minimize token usage while staying reliable.

In [11]:
# ── Pure Python preprocessor — replaces all 5 specialist agents ──────────────
# No LLM calls here. Builds a compact evidence dict per user for the scorer.

def preprocess_user(sender_id: str, dataset: dict) -> dict:
    """
    Runs all analysis in pure Python — zero LLM calls.
    Returns a compact evidence dict ready for FraudScorerAgent.
    """
    df        = dataset["transactions"]
    users     = dataset["users"]
    sms_data  = dataset["sms"]
    mail_data = dataset["mails"]
    locations = dataset["locations"]

    user_txns = df[df["sender_id"] == sender_id].sort_values("timestamp")
    if user_txns.empty:
        return {"sender_id": sender_id, "transactions": [], "phishing": [], "profile": {}}

    # ── Salary reference ──────────────────────────────────────────────────────
    iban_salary = {u["iban"]: u["salary"] for u in users if "iban" in u}
    sender_iban = user_txns["sender_iban"].dropna().iloc[0] \
                  if not user_txns["sender_iban"].dropna().empty else None
    monthly_salary = (iban_salary.get(sender_iban, 0) or 0) / 12

    # ── User profile ──────────────────────────────────────────────────────────
    user = next((u for u in users if u.get("iban") == sender_iban), None)
    if not user:
        parts = sender_id.split("-")
        user = next((u for u in users
                     if len(parts) >= 2 and (
                         parts[1][:3].upper() in u.get("first_name","").upper()[:4] or
                         parts[0][:3].upper() in u.get("last_name","").upper()[:4]
                     )), None)
    first_name = user.get("first_name", sender_id.split("-")[0]) if user else sender_id.split("-")[0]

    # Phishing risk from description keywords
    desc = (user.get("description","") if user else "").lower()
    pct  = re.search(r"(\d+)\s*%", desc)
    if pct and int(pct.group(1)) >= 50:
        phishing_risk = "high"
    elif any(k in desc for k in ["leichtgläubig","crédule","not always careful","clicks","neugierig"]):
        phishing_risk = "high"
    elif any(k in desc for k in ["prudent","vorsichtig","careful","méfiant"]):
        phishing_risk = "low"
    else:
        phishing_risk = "medium"

    # ── Transaction signals ───────────────────────────────────────────────────
    times = user_txns["timestamp"].tolist()
    ids   = user_txns["transaction_id"].tolist()

    # Rapid sequence
    rapid_ids = set()
    for i in range(len(times)):
        w = [ids[j] for j in range(i, len(times))
             if times[j] - times[i] <= timedelta(hours=1)]
        if len(w) >= 3:
            rapid_ids.update(w)

    # Balance drain
    median_bal = user_txns["balance_after"].median()
    drain_ids  = set(
        user_txns[user_txns["balance_after"] < median_bal * 0.05]["transaction_id"]
    ) if median_bal > 0 else set()

    # Build transaction list with flags
    seen, txn_list = set(), []
    for _, r in user_txns.iterrows():
        rid   = r.get("recipient_id")
        flags = []
        if r["timestamp"].hour in range(0, 5):          flags.append("night")
        if pd.notna(rid) and rid not in seen:            flags.append("new_recipient")
        if r["transaction_id"] in rapid_ids:             flags.append("rapid_sequence")
        if r["transaction_id"] in drain_ids:             flags.append("balance_drain")
        if r["transaction_type"] == "e-commerce" and r["timestamp"].hour in range(0,5):
                                                         flags.append("ecommerce_night")
        # Skip obviously legitimate transactions
        desc_str = str(r.get("description","")).lower()
        is_legit = any(k in desc_str for k in
                       ["salary","rent payment","rent payment jan","rent payment feb",
                        "rent payment mar","rent payment apr","rent payment may",
                        "rent payment jun","rent payment jul","rent payment aug",
                        "rent payment sep","rent payment oct","rent payment nov",
                        "rent payment dec"])

        ratio = round(r["amount"] / monthly_salary, 2) if monthly_salary else 0
        if ratio > 1.5 and not is_legit:                 flags.append(f"high_amount:{ratio}x")

        if pd.notna(rid):
            seen.add(rid)

        if flags and not is_legit:
            txn_list.append({
                "id":     r["transaction_id"],
                "hour":   r["timestamp"].hour,
                "amount": r["amount"],
                "type":   r["transaction_type"],
                "desc":   desc_str[:60],
                "flags":  flags,
            })

    # ── Phishing signals ──────────────────────────────────────────────────────
    PHISHING_DOMAINS   = ["amaz0n","paypa1","ub3r","netfl1x","r1d3share","citydriv3"]
    PHISHING_KEYWORDS  = ["urgent","verify","suspend","unusual login","account locked"]

    phishing_events = []
    for s in sms_data:
        text = s.get("sms","")
        if first_name.lower() not in text.lower():
            continue
        text_lower = text.lower()
        dom_hits = [d for d in PHISHING_DOMAINS if d in text_lower]
        kw_hits  = [k for k in PHISHING_KEYWORDS if k in text_lower]
        if dom_hits or kw_hits:
            lines    = text.split("\n")
            date_str = next((l.replace("Date:","").strip() for l in lines if l.startswith("Date:")), "")
            try:    ts = str(pd.to_datetime(date_str))
            except: ts = ""
            phishing_events.append({
                "channel":  "sms",
                "ts":       ts,
                "severity": "high" if dom_hits else "medium",
                "domain":   dom_hits[0] if dom_hits else "",
            })

    for m in mail_data:
        text = m.get("mail","")
        if first_name.lower() not in text.lower():
            continue
        text_lower = text.lower()
        dom_hits = [d for d in PHISHING_DOMAINS if d in text_lower]
        kw_hits  = [k for k in PHISHING_KEYWORDS if k in text_lower]
        if dom_hits or kw_hits:
            lines    = text.split("\n")
            date_str = next((l.replace("Date:","").strip() for l in lines if l.startswith("Date:")), "")
            try:    ts = str(pd.to_datetime(date_str, utc=True))
            except: ts = ""
            phishing_events.append({
                "channel":  "email",
                "ts":       ts,
                "severity": "high" if dom_hits else "medium",
                "domain":   dom_hits[0] if dom_hits else "",
            })

    # ── Location signals ──────────────────────────────────────────────────────
    ip_txns = user_txns[user_txns["transaction_type"] == "in-person payment"]
    location_flags = []
    if not ip_txns.empty:
        user_locs = [l for l in locations
                     if any(p in l.get("biotag","") for p in sender_id.split("-")[:2])]
        for _, r in ip_txns.iterrows():
            txn_city = str(r.get("location","")).split("-")[0].strip().lower()
            nearby   = [l for l in user_locs
                        if abs((pd.to_datetime(r["timestamp"]) -
                                pd.to_datetime(l["timestamp"])).total_seconds()) < 7200]
            if nearby:
                gps_city = nearby[0].get("city","").lower()
                if gps_city and txn_city and gps_city not in txn_city and txn_city not in gps_city:
                    location_flags.append({
                        "txn_id":   r["transaction_id"],
                        "txn_city": txn_city,
                        "gps_city": gps_city,
                    })

    # ── Phishing → transaction correlation (72h window) ──────────────────────
    for txn in txn_list:
        try:    txn_time = pd.to_datetime(txn["id"])
        except: txn_time = None
        preceding = []
        for evt in phishing_events:
            try:
                evt_time   = pd.to_datetime(evt["ts"])
                txn_time_r = user_txns[user_txns["transaction_id"]==txn["id"]]["timestamp"].iloc[0]
                hours_gap  = (txn_time_r - evt_time).total_seconds() / 3600
                if 0 <= hours_gap <= 72:
                    preceding.append(round(hours_gap,1))
            except: pass
        if preceding:
            txn["phishing_before_hours"] = min(preceding)

    return {
        "sender_id":      sender_id,
        "first_name":     first_name,
        "phishing_risk":  phishing_risk,
        "monthly_salary": monthly_salary,
        "transactions":   txn_list,           # only flagged ones
        "phishing":       phishing_events,
        "location_flags": location_flags,
    }

print("Preprocessor ready.")

Preprocessor ready.


## Cell 7 — @tool Wrappers
Each specialist is wrapped as a `@tool` so the FraudScorerAgent can call them.
Calling pattern mirrors the tutorial:
```python
response = agent.invoke({"messages": [HumanMessage(...)]})
return response["messages"][-1].content
```

In [12]:
# ── FraudScorerAgent — 1 LLM call per user, no tools ─────────────────────────
# All data already preprocessed. Agent just reasons and decides.

fraud_scorer_agent = create_agent(
    model=model,
    system_prompt="""You are a fraud investigator for MirrorPay.
You receive pre-analyzed evidence and decide which transactions are fraudulent.

DECISION RULES:
- Flag if: night transaction + new recipient + phishing context → fraudulent
- Flag if: balance drain + high amount → fraudulent  
- Flag if: rapid sequence + new recipient → fraudulent
- Flag if: e-commerce at night + no description + new recipient → fraudulent
- NEVER flag: salary, rent, phone bills, subscriptions, internet bills
- NEVER flag: direct debits with plan/subscription/bill in description
- New recipient alone → NOT enough, need 2+ signals
- Low amount (<5% monthly salary) at night → likely automated, skip

Return ONLY this JSON, nothing else:
{"sender_id":"...","fraudulent_transactions":["id1","id2"],"reasoning":"max 20 words"}"""
)

print("FraudScorerAgent ready.")

FraudScorerAgent ready.


## Cell 8 — FraudScorerAgent (Orchestrator)
Uses `create_agent(model=model, system_prompt="...", tools=[...])` — exact tutorial pattern.
This is the **only agent that calls the LLM as an orchestrator**.

In [13]:
# ── FraudScorerAgent — 1 LLM call per user, no tools ─────────────────────────

fraud_scorer_agent = create_agent(
    model=model,
    system_prompt="""You are a fraud investigator for MirrorPay.
You receive pre-analyzed evidence and decide which transactions are fraudulent.

DECISION RULES:
- Flag if: night transaction + new recipient + phishing context → fraudulent
- Flag if: balance drain + high amount → fraudulent
- Flag if: rapid sequence + new recipient → fraudulent
- Flag if: e-commerce at night + no description + new recipient → fraudulent
- NEVER flag: salary, rent, phone bills, subscriptions, internet bills
- NEVER flag: direct debits with plan/subscription/bill in description
- New recipient alone → NOT enough, need 2+ signals
- Low amount (<5% monthly salary) at night → likely automated, skip

Return ONLY this JSON, nothing else:
{"sender_id":"...","fraudulent_transactions":["id1","id2"],"reasoning":"max 20 words"}"""
)

print("FraudScorerAgent ready.")

FraudScorerAgent ready.


## Cell 9 — Run per User + Orchestrator

In [14]:
def resolve_first_name(sender_id: str, dataset: dict) -> str:
    """Resolves user first name from sender_id via IBAN match or abbreviation fallback."""
    df, users = dataset["transactions"], dataset["users"]
    sender_txns = df[df["sender_id"] == sender_id]
    sender_iban = sender_txns["sender_iban"].dropna().iloc[0] \
                  if not sender_txns["sender_iban"].dropna().empty else None

    # Strategy 1: IBAN match
    iban_map = {u["iban"]: u.get("first_name", "") for u in users if "iban" in u}
    if sender_iban and sender_iban in iban_map:
        return iban_map[sender_iban]

    # Strategy 2: sender_id abbreviation match
    parts = sender_id.split("-")
    if len(parts) >= 2:
        for u in users:
            fn = u.get("first_name", "").upper()
            ln = u.get("last_name",  "").upper()
            if parts[1][:3] in fn[:4] or parts[0][:3] in ln[:4]:
                return u.get("first_name", "")

    return parts[0].capitalize()


def run_fraud_scorer(sender_id: str, user_first_name: str,
                     session_id: str, evidence: dict) -> dict:
    """1 LLM call per user — evidence already preprocessed in Python."""
    langfuse_handler = get_callback_handler()

    prompt = (
        f"Sender: {sender_id} ({user_first_name}) | "
        f"Phishing risk: {evidence['phishing_risk']} | "
        f"Monthly salary: {evidence['monthly_salary']:.0f}\n"
        f"Phishing attacks: {len(evidence['phishing'])} "
        f"({sum(1 for p in evidence['phishing'] if p['severity']=='high')} high severity)\n"
        f"Location anomalies: {len(evidence['location_flags'])}\n"
        f"Suspicious transactions ({len(evidence['transactions'])} total):\n"
        + json.dumps(evidence["transactions"], indent=2)
    )

    response = fraud_scorer_agent.invoke(
        {"messages": [HumanMessage(prompt)]},
        config={
            "callbacks": [langfuse_handler],
            "metadata":  {"langfuse_session_id": session_id},
        }
    )

    final_text = response["messages"][-1].content
    try:
        m = re.search(r"```json\s*(.*?)\s*```", final_text, re.DOTALL)
        parsed = json.loads(m.group(1) if m else final_text)
    except Exception:
        try:
            m = re.search(r"\{.*\}", final_text, re.DOTALL)
            parsed = json.loads(m.group(0)) if m else {}
        except Exception:
            parsed = {}

    return {
        "sender_id":               sender_id,
        "fraudulent_transactions": parsed.get("fraudulent_transactions", []),
        "reasoning":               parsed.get("reasoning", final_text[:100]),
    }


def run_orchestrator(dataset: dict) -> list:
    """Main entry point — investigates all citizens and writes output file."""
    session_id = generate_session_id()
    print(f"Session : {session_id}")
    print(f"Level   : {dataset['level']}")
    print("-" * 60)

    df = dataset["transactions"]
    SKIP = ("EMP", "ABIT", "DOM", "BTSWF")
    senders = [
        s for s in df["sender_id"].unique()
        if not any(str(s).startswith(p) for p in SKIP)
    ]

    print(f"Investigating {len(senders)} citizen(s)...\n")
    results = []

    for i, sid in enumerate(senders, 1):
        # Step 1: pure Python — no LLM
        evidence = preprocess_user(sid, dataset)
        name     = evidence["first_name"]
        print(f"[{i}/{len(senders)}] {sid} ({name}) — "
              f"{len(evidence['transactions'])} suspicious txns, "
              f"{len(evidence['phishing'])} phishing events")

        # Step 2: 1 LLM call
        result = run_fraud_scorer(sid, name, session_id, evidence)
        results.append(result)

        flagged = result.get("fraudulent_transactions", [])
        print(f"  Flagged : {len(flagged)} | {result.get('reasoning','')[:100]}")
        for fid in flagged:
            print(f"    - {fid}")
        print()

    # Write output file
    fraud_ids, seen = [], set()
    for r in results:
        for fid in r.get("fraudulent_transactions", []):
            if fid not in seen:
                seen.add(fid)
                fraud_ids.append(fid)

    out_path = "/kaggle/working/fraudulent_transactions.txt"
    with open(out_path, "w") as f:
        f.write("\n".join(fraud_ids))

    print("=" * 60)
    print(f"Output   : {out_path}")
    print(f"Flagged  : {len(fraud_ids)} transactions")

    langfuse_client.flush()
    print(f"Traces   : sent to Langfuse | session: {session_id}")

    return results


print("run_orchestrator ready.")

run_orchestrator ready.


## Cell 10 — Run Pipeline

In [15]:
results = run_orchestrator(DATA)

Session : Hashtag-01KPBRJXVCCG8EA10YDW5KDRV6
Level   : brave_new_world
------------------------------------------------------------
Investigating 7 citizen(s)...

[1/7] PTLA-CHRS-7FD-POR-1 (Christopher) — 23 suspicious txns, 12 phishing events
  Flagged : 6 | Night transactions with new recipients and phishing context.
    - 65fdecad-2431-4ad8-affa-4efb655eaf7f
    - a24ebe1b-a588-4f51-a1a3-c779698735ae
    - 800445ca-2686-4c49-a4d7-5386b85d3c98
    - e1b3bc30-9ec3-440e-9d5c-d0f7c67dd5ed
    - d6b436f9-ff10-4cdb-b99f-7308f6a4643c
    - 01e089cd-dbe1-46da-89d9-ec675702e68b

[2/7] MRRS-DBRH-800-LON-1 (MRRS) — 28 suspicious txns, 0 phishing events
  Flagged : 6 | Multiple signals of new recipients and night transactions.
    - 8aa71d57-b166-4a86-994e-052baa474095
    - 678c5d9f-3c1a-402d-bdcc-2b96ab6a83a2
    - b20baeac-58a2-4e64-9264-d14fc98847f2
    - f85ecf4f-fad5-459e-af1f-43aa0edb3c91
    - 8d09eadd-dc11-4d40-8bcc-abc308b1c8cb
    - fe94ab33-4899-4c62-955b-aef1f1579e37

[3/7] NVRR-LX

## Cell 11 — Validate Output
Checks the output file against the challenge validity rules.

In [16]:
with open("/kaggle/working/fraudulent_transactions.txt") as f:
    flagged_ids = [line.strip() for line in f if line.strip()]

all_ids     = DATA["transactions"]["transaction_id"].tolist()
pct_flagged = len(flagged_ids) / len(all_ids) * 100

print(f"Total transactions : {len(all_ids)}")
print(f"Flagged as fraud   : {len(flagged_ids)}")
print(f"Percentage flagged : {pct_flagged:.1f}%")
print()

# Challenge validity checks
assert len(flagged_ids) > 0,            "INVALID: no transactions reported"
assert len(flagged_ids) < len(all_ids), "INVALID: all transactions reported"

invalid_ids = [fid for fid in flagged_ids if fid not in all_ids]
if invalid_ids:
    print(f"WARNING: {len(invalid_ids)} invalid IDs found:")
    for bad in invalid_ids:
        print(f"  {bad}")
else:
    print("All flagged IDs are valid. Output is ready to submit.")

print("\nFlagged transaction IDs:")
for fid in flagged_ids:
    print(f"  {fid}")

Total transactions : 524
Flagged as fraud   : 27
Percentage flagged : 5.2%

All flagged IDs are valid. Output is ready to submit.

Flagged transaction IDs:
  65fdecad-2431-4ad8-affa-4efb655eaf7f
  a24ebe1b-a588-4f51-a1a3-c779698735ae
  800445ca-2686-4c49-a4d7-5386b85d3c98
  e1b3bc30-9ec3-440e-9d5c-d0f7c67dd5ed
  d6b436f9-ff10-4cdb-b99f-7308f6a4643c
  01e089cd-dbe1-46da-89d9-ec675702e68b
  8aa71d57-b166-4a86-994e-052baa474095
  678c5d9f-3c1a-402d-bdcc-2b96ab6a83a2
  b20baeac-58a2-4e64-9264-d14fc98847f2
  f85ecf4f-fad5-459e-af1f-43aa0edb3c91
  8d09eadd-dc11-4d40-8bcc-abc308b1c8cb
  fe94ab33-4899-4c62-955b-aef1f1579e37
  bf2e6c3f-ac15-4b1a-a0fb-c1578ffbacb5
  152b88bd-d570-472f-a24c-5d33d3e9a116
  527db4a0-67c9-4af7-9dc8-c664d7431bd9
  b01db2d3-a5cf-48fa-8fc9-f3d3e43ee1a9
  4ab41abb-3cfa-4e87-96f2-1dd8b0852b52
  9e6b1148-a8e6-406c-a2af-2c81a21cef31
  3f853f1f-925e-4eb8-ac57-3fc3d93189c4
  5d8d02fc-aea1-4065-a33d-7acc4b385de6
  a3f9c85d-f360-4f3c-81d3-330cabb3e580
  a1b1f4e7-96e7-4a08-bfae